In [43]:
import pandas as pd
df = pd.read_csv('../data/raw/tech_layoffs_til_2025.csv')

Standardize "United Kingdom" to "UK" since UK is the more common spelling 
in this dataset (74 vs 5 occurrences).

In [44]:
df['Country'] = df['Country'].replace('United Kingdom', 'UK')

Industry column has 118 categories; many appear only 1-2 times. 
Grouping low-frequency categories (appearing fewer than 5 times) into 
"Other" to make industry-level analysis more meaningful.

In [45]:
industry_counts = df['Industry'].value_counts()
rare_industries = industry_counts[industry_counts < 5].index
df['Industry'] = df['Industry'].apply(lambda x: 'Other' if x in rare_industries else x)

In [46]:
df['Industry'].value_counts()

Industry
Finance                          289
Other                            228
Retail                           176
Healthcare                       156
Transportation                   142
Consumer                         134
Food                             133
Marketing                        121
HR                                79
Security                          79
Education                         77
Real Estate                       74
Media                             74
Data                              66
Crypto                            65
Travel                            52
Software Development              44
Logistics                         44
Sales                             41
Infrastructure                    37
Energy                            32
Product                           30
Hardware                          29
Support                           29
Fitness                           22
AI                                17
Gaming                       

"Unknown" in the Stage column (356 records) carries no usable funding-stage 
information. Converting it to a proper missing value (NaN) so it's handled 
consistently with other missing data during analysis.

In [47]:
import numpy as np
df['Stage'] = df['Stage'].replace('Unknown', np.nan)

In [48]:
# To verify if there's still 'Unknown' in 'Stage' column
df['Stage'].value_counts()

Stage
Post-IPO          574
Series B          266
Series C          211
Series D          203
Acquired          176
Series E          123
Series A          116
Series F           66
Seed               49
Private Equity     37
Series H           27
Series G           19
Subsidiary         11
Series I            8
Series J            6
Name: count, dtype: int64

Region column is too coarse for analysis (67% classified as "other") and 
is being dropped. Country will be used instead for geographic analysis.

In [49]:
df.drop(columns = ['Region'])

,Nr,Company,Location_HQ,USState,Country,Continent,Laid_Off,Date_layoffs,Percentage,Company_Size_before_Layoffs,Company_Size_after_layoffs,Industry,Stage,Money_Raised_in__mil,Year,latitude,longitude
0,1,Tamara Mellon,Los Angeles,California,USA,North America,20.0,2020-03-12,40.0,50.0,30.0,Retail,Series C,90.0,2020,34.053691,-118.242766
1,2,HopSkipDrive,Los Angeles,California,USA,North America,8.0,2020-03-13,10.0,80.0,72.0,Transportation,NaN,45.0,2020,34.053691,-118.242766
2,3,Panda Squad,San Francisco,California,USA,North America,6.0,2020-03-13,75.0,8.0,2.0,Consumer,Seed,1.0,2020,37.779259,-122.419329
3,4,Help.com,Austin,Texas,USA,North America,16.0,2020-03-16,100.0,16.0,0.0,Support,Seed,6.0,2020,30.271129,-97.743700
4,5,Inspirato,Denver,Colorado,USA,North America,130.0,2020-03-16,22.0,591.0,461.0,Travel,Series C,79.0,2020,39.739236,-104.984862
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,2408,Amazon,Seattle,Washington,USA,North America,84.0,2025-12-15,NaN,NaN,NaN,E-commerce,NaN,NaN,2025,47.603832,-122.330062
2408,2409,The Trade Desk,Ventura,California,USA,North America,NaN,2025-12-17,1.0,NaN,NaN,Other,NaN,NaN,2025,34.281144,-119.214061
2409,2410,Amazon,Luxembourg,non,Luxembourg,Europe,370.0,2025-12-19,NaN,NaN,NaN,E-commerce,NaN,NaN,2025,49.609567,6.133575
2410,2411,Yellow.ai,San Mateo,California,USA,North America,100.0,2025-12-23,NaN,NaN,NaN,Other,NaN,NaN,2025,37.496904,-122.333057


Binning Company_Size_before_Layoffs into 3 categories based on the quantile 
analysis from the exploration phase: Small (≤300), Mid-size (301-1000), 
Large (>1000).

In [50]:
df['Company_Size_Category'] = pd.cut(
    df['Company_Size_before_Layoffs'],
    bins=[0, 300, 1000, 99999999999],
    labels=['Small', 'Mid-size', 'Large']
)

In [51]:
df['Company_Size_Category'].value_counts()

Company_Size_Category
Small       587
Mid-size    585
Large       584
Name: count, dtype: int64

Dropping `Nr` (a row index with no analytical value) and `Continent` 
(too coarse, overlaps with Country which is already used for geographic 
analysis).

In [52]:
df = df.drop(columns=['Nr', 'Continent'])

In [53]:
df.to_csv('../data/cleaned/tech_layoffs_cleaned.csv', index=False)